# <span style="color: red; font-weight:bold"> Multivariate Regime-Conditional Simulator 
## <span style="color: red; font-weight:bold"> Simulated World Project.
### <span style="color: brown; font-weight:bold"> Part 1.1: Synthetic Worlds. Data Collection and Features Engineering.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For reading stock data from yahoo
import yfinance as yf
# For time stamps
from datetime import datetime, date, timedelta
# For reading data from FED
from fredapi import Fred
import requests

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [2]:
x = None
print(x == None, x is None)

True True


## <span style="color: blue; font-weight:bold"> Data: 
### <span style="color:green; font-weight:bold">YAHOO Finance source:
        Target  -> S&P500 iindex daily price (2000 - 2025), '^GSPC', Column: 'Close'
        Feature -> S&P500 iindex daily daily volume,        '^GSPC', Column: 'Volume_Change'
        Feature -> Marcket Volatility Indicator,            '^VIX',  Column: 'VIX_Close'
        'UNRATE' + 'PAYEMS': Monthly, first Friday, ~1 week after reference month ends
        'GDP':                           Quarterly, ~4 weeks after quarter ends
        'CPIAUCSL' (CPI):       Monthly, mid-month, ~2 weeks after reference month ends

### <span style="color:green; font-weight:bold">FRED source. Key Indicators for Stock Market Prediction:
        Feature -> 10-Year Treasury Rate (Interest Rates),      'DGS10',  Column: 'DGS10_Change'
        Feature -> 10-Year minus 2-Year Treasury (Yield Curve), 'T10Y2Y', Column: 'T10Y2Y_Change' 
        Feature -> 13-WEEK TB,                                  'DTB3',       

    -----------------------------------------------------------------------------------------------------------
    
    FRED source. Key Indicators
    
    'DGS10':  '10-Year Treasury Rate (Interest Rates)',       # Daily https://fred.stlouisfed.org/series/DGS10
    10-year U.S. Treasury note, quoted on an investment basis. It is a key indicator for long-term borrowing costs, mortgage rates, and investor confidence in future growth.
   
    'T10Y2Y': '10-Year minus 2-Year Treasury (Yield Curve)',  # Daily https://fred.stlouisfed.org/series/T10Y2Y
   
    'DTB3':   '13-WEEK TB',                                   # Daily https://fred.stlouisfed.org/series/dtb3
    the interest rate on a 3-month U.S. Treasury bill, quoted on a discount basis. It is a short-term, low-risk, highly liquid rate that acts as a proxy for immediate cash-like investment

    'PAYEMS': the total number of paid employees in the non-agriculture sector in the U.S.
    'observation_period', 
    'as_of_date_release', 
    '{indicator}_value'

    'UNRATE': The unemployment rate represents the number of unemployed as a percentage of the labor force.
    'observation_period', 
    'as_of_date_release', 
    '{indicator}_value'
    
    'GDP': Gross domestic product (GDP), the featured measure of U.S. output, is the market value of the goods 
           and services produced by labor and property located in the United States.
    'observation_period', 
    'as_of_date_release', 
    '{indicator}_value'
    
    'CPIAUCSL': (CPI) The Consumer Price Index for All Urban Consumers: All Items (CPIAUCSL) is a price index of a basket of goods 
                and services paid by urban consumers. Percent changes in the price index measure the inflation rate between any two time periods.
    'observation_period', 
    'as_of_date_release', 
    '{indicator}_value'
    -----------------------------------------------------------------------------------------------------------
    news_sentiment_data.csv  https://www.frbsf.org/research-and-insights/data-and-indicators/daily-news-sentiment-index/
    The Daily News Sentiment Index is a high frequency measure of economic sentiment based on lexical analysis of economics-related news articles. 
    The index is described in Buckman,   Shapiro, Sudhof, and Wilson (2020) and based on the methodology developed in Shapiro, Sudhof, and Wilson (2020).

In [3]:
################ Check if any NaN exists in the entire DataFrame ##########################
def check_nan(df):
    nan_mask = df.isna()
    has_nan = nan_mask.any().any()
    print(f"Does the DataFrame contain any NaN values? {has_nan}")
    if has_nan:
        nan_counts = nan_mask.sum()
        print(f"Count of NaN values per column:\n{nan_counts[nan_counts > 0]}")
###########################################################################################

# <span style='color:brown'> Get Historical Data from YAHOO FINANCE 

In [4]:
################## Get Price History from YAHOO FINANCE #################
def get_yf_data(ticker, start, end):
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(start=start, end=end)
        if data.empty:
            print(f"No data found for {ticker} ticker")
            return None
        print(f"Successerfully dowloaded data for {ticker}")
        print(f"Data shape: {data.shape}")
        return data

    except Exception as e:
        print(f"Error dowloading data for {ticker}: {e}")
        return None

###############################################################################

In [5]:
# Dictinory TICKER: FILE_NAME
yahoo_tickers_csv = {
    '^GSPC': 'yfinance_i_sp500',  # S&P 500 Index
    'DX-Y.NYB': 'yfinance_i_dxy', # Dolar Index
    '^VIX': 'yfinance_i_vix',     # Scare Index
    }

In [6]:
#===========================================================================
# FOR DATA COLLECTION AND MODEL TRAINING 
#===========================================================================
# Set start day and end day data collection
start_date = '2000-01-01'
end_date = '2026-01-01'

In [7]:
print(f"End day: {end_date}, To get data UP TO end_date (inclusive), request end_date + 1")
print("Start day:", start_date)
print("End day", end_date)

End day: 2026-01-01, To get data UP TO end_date (inclusive), request end_date + 1
Start day: 2000-01-01
End day 2026-01-01


In [8]:
# Save data sets
for ticker, name in yahoo_tickers_csv.items():
    yf_data = get_yf_data(ticker, start_date, end_date)
    yf_data.to_csv(f"project_DATA/YF_data_{name}.csv")

Successerfully dowloaded data for ^GSPC
Data shape: (6539, 7)
Successerfully dowloaded data for DX-Y.NYB
Data shape: (6568, 7)
Successerfully dowloaded data for ^VIX
Data shape: (6539, 7)


In [9]:
yf_data_list_name = ['yf_sp500_data_index', 'yf_dxy_data_index', 'yf_vix_data_index']

In [10]:
#####################    ##################################
# Convert to datetime with UTC handling, then drop timezone
def convert_to_datetime(df):
    df.index = pd.to_datetime(df.index, utc=True).tz_convert(None)
    df.index = df.index.normalize()  # remove 05:00:00 time
    return df
###########################################################

In [11]:
# Create a YF data sets dictionary
yf_data_dict = {}

# Get data with the required columns
i = 0
for ticker, name in yahoo_tickers_csv.items():
    if ticker not in ['^GSPC']:
        # DXY & VIX : Date & Close
        yf_data_dict[i] = pd.read_csv(f"project_DATA/YF_data_{name}.csv", parse_dates=['Date']).set_index('Date')[['Close']]
    else:
        # For S&P_500 INDX : Date & HLOC
        yf_data_dict[i] = pd.read_csv(f"project_DATA/YF_data_{name}.csv", parse_dates=['Date']).set_index('Date').drop(columns=['Dividends', 'Stock Splits'])
        
    i +=1

# # Convert to datetime and rename columns
# 1
yf_sp500_data_index_df = convert_to_datetime(yf_data_dict[0])
# 2
yf_dxy_data_index_df = convert_to_datetime(yf_data_dict[1].rename(columns={'Close': 'DXY_indx_close'}))
# 3
yf_vix_data_index_df = convert_to_datetime(yf_data_dict[2].rename(columns={'Close': 'VIX_indx_close'}))

# <span style="color:brown"> FRED API data source

In [12]:
# Use actual FRED API key
with open('project_DATA/fredkey.txt', 'r', encoding='utf-8') as file:
    content = file.read()
    
FRED_API_KEY = content
fred = Fred(api_key=FRED_API_KEY)

### <span style="color:orange"> 1. FRED FINANCE data

In [13]:
# Key Indicators for Stock Market Prediction
KEY_INDICATORS = {
    'DGS10':  '10-Year Treasury Rate (Interest Rates)',        # Daily https://fred.stlouisfed.org/series/DGS10
    'T10Y2Y': '10-Year minus 2-Year Treasury (Yield Curve)',   # Daily https://fred.stlouisfed.org/series/T10Y2Y
    'DTB3':   '13-WEEK TB',                                    # Daily https://fred.stlouisfed.org/series/dtb3
}

In [14]:
#############################################################################
def get_fred_data(series_id, start_date, end_date):
    """
    Fetch data from FRED API
    """
    url = f"https://api.stlouisfed.org/fred/series/observations?series_id=GNPCA&api_key=abcdefghijklmnopqrstuvwxyz123456&file_type=json"

    # Data to collect
    params = {
        'series_id': series_id,
        'api_key': FRED_API_KEY,
        'file_type': 'json',
        'observation_start': start_date,
        'observation_end': end_date,
        'frequency': 'd'  # Daily where available
    }
    
    try:
        response = requests.get(url, params=params)
        data = response.json()
        
        # Convert to DataFrame
        df = pd.DataFrame(data['observations'])
        df['date'] = pd.to_datetime(df['date'])
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        df = df[['date', 'value']].set_index('date')
        df.columns = [series_id]
        
        return df
    except Exception as e:
        print(f"Error fetching {series_id}: {e}")
        return pd.DataFrame()
#############################################################################

In [15]:
################## 1. Get FRED FINANCE data COMBINED #############################

def get_finance_fred_features(start_date, end_date):
    fred_data = {}
    n = 4
    # Fetch all indicators
    for indicator_id, description in KEY_INDICATORS.items():      
        print(f"# {n}")
        print(f"Fetching {indicator_id}: {description}")
        n +=1
    # Get the data
        data = get_fred_data(indicator_id, start_date, end_date)
        if not data.empty:
            fred_data[indicator_id] = data
        else:
            print(f"Warning: No data for {indicator_id}")
    
    # Combine all economic data
    if fred_data:
        # Merge all dataframes
        combined_fred_data = pd.concat(fred_data.values(), axis=1)
        combined_fred_data = combined_fred_data.sort_index()
        
        # Handle missing values - forward fill for daily data
        combined_fred_data = combined_fred_data.ffill()  # Carry forward last known value
        
        return combined_fred_data
    else:
        return pd.DataFrame()
############################################################################

In [16]:
# FRED data included the end_date data for collection

# Convert str date to datetime
datetime_end_date = pd.to_datetime(end_date)
# One day less to alligne with YF data
fred_end_date = datetime_end_date - timedelta(days=1)
# Convert back to the str
fred_end_date_str = fred_end_date.strftime("%Y-%m-%d")

In [17]:
# Get the FRED data combined and date aligned
fred_fin_data = get_finance_fred_features(start_date, fred_end_date_str) 

# 4
Fetching DGS10: 10-Year Treasury Rate (Interest Rates)
# 5
Fetching T10Y2Y: 10-Year minus 2-Year Treasury (Yield Curve)
# 6
Fetching DTB3: 13-WEEK TB


### <span style="color:green">  NEW FEATURES U.S. Treasury

In [18]:
# Rename index
fred_fin_data.index.name = 'Date'
# Save a copy
fred_fin_data_df = fred_fin_data.copy(deep=True)

In [19]:
# U.S. Treasury new features
def new_features_fred_fin(data_df):
    df = data_df.copy(deep=True)
    df['term_spread'] = df['DGS10'] - df['T10Y2Y']
    df['T10Y3M'] = df['DGS10'] - df['DTB3']
    
    # Recession signal
    df['T10Y2Y_yield_curve'] = (df['T10Y2Y'] <0).astype(int)
    
    names_fred_fin = ['DGS10', 'T10Y2Y', 'DTB3']
    for name in names_fred_fin:
        df[f"{name}_dif"] = df[f"{name}"].diff()

    return df

In [20]:
# Create a data set with new FRED-based features
fred_fin_data_df = new_features_fred_fin(fred_fin_data_df)

In [21]:
fred_fin_data_df.head(3)

,DGS10,T10Y2Y,DTB3,term_spread,T10Y3M,T10Y2Y_yield_curve,DGS10_dif,T10Y2Y_dif,DTB3_dif
Date,,,,,,,,,
2000-01-03,6.58,0.20,5.27,6.38,1.31,0,NaN,NaN,NaN
2000-01-04,6.49,0.19,5.27,6.30,1.22,0,-0.09,-0.01,0.00
2000-01-05,6.62,0.24,5.28,6.38,1.34,0,0.13,0.05,0.01


### <span style="color:orange"> 2. FRED ECONOMIC data

In [22]:
############ Get first release data for economic indicators from FRED ###############
def get_first_release(indicator):
    """
    Parameters:
        indicator (str): FRED series ID ('GDP', 'CPIAUCSL', 'PAYEMS', 'UNRATE')
    Returns:
        pd.DataFrame: First release data with as_of_date as index
    """
    # Get all realeses - this shows the date when each data point was released GDP CPIAUCSL PAYEMS UNRATE
    all_releases = fred.get_series_all_releases(indicator)
    all_data = all_releases.copy()
    # Rename columns for clarity
    all_data.rename(columns={
        'date': f'{indicator}_observation_period',
        'realtime_start': 'as_of_date_release',  # When we knew this value
        'value': f'{indicator}_value',
    },  
        inplace=True)
    
    # Ensure dates are datetime
    all_data['as_of_date_release'] = pd.to_datetime(all_data['as_of_date_release'])

    all_data[f'{indicator}_observation_period'] = pd.to_datetime(all_data[f'{indicator}_observation_period'])

    # Ensure value column is numeric
    all_data[f'{indicator}_value'] = pd.to_numeric(all_data[f'{indicator}_value'], errors='coerce')

    # Get first release for each observation period
    indicator_first_release = all_data.groupby(f'{indicator}_observation_period').first().reset_index()
    
    # Proper sorting. Sort by observation period
    indicator_first_release = indicator_first_release.sort_values(f'{indicator}_observation_period')

    # Set as_of_date as index and sort by it
    indicator_first_release = indicator_first_release.set_index('as_of_date_release').sort_index()

    return indicator_first_release    
###############################################################################

In [23]:
# Get all historical data for each indicator
gdp_data = get_first_release('GDP')
cpi_data = get_first_release('CPIAUCSL')
payems_data = get_first_release('PAYEMS')
unrate_data = get_first_release('UNRATE')

In [24]:
# Create a dictionary
data_dict = {
    'gdp_all_data': gdp_data,
    'cpi_all_data': cpi_data,
    'payems_all_data': payems_data,
    'unrate_all_data': unrate_data
    }

In [25]:
# Save the FRED ECONOMIC data to .csv
for name, data in data_dict.items():
    data.to_csv(f"project_DATA/data_fred_{name}.csv")

In [26]:
# Directly expand the dictionary keys into a list.
data_dict_keys_list = [*data_dict]

In [27]:
# Get the saved data and setup the index column
fred_data_dict = {}
i = 0
for ind_name in data_dict_keys_list:
    fred_data_dict[i] = pd.read_csv(f"project_DATA/data_fred_{ind_name}.csv", parse_dates=['as_of_date_release']).set_index('as_of_date_release')       
    i +=1
    

In [28]:
# Save the ECONOMIC data DF and set the datetime
# 7
fred_gdp_data_df = fred_data_dict[0]
fred_gdp_data_df['GDP_observation_period'] = pd.to_datetime(fred_gdp_data_df['GDP_observation_period'])
# 8
fred_cpi_data_df = fred_data_dict[1]
fred_cpi_data_df['CPIAUCSL_observation_period'] = pd.to_datetime(fred_cpi_data_df['CPIAUCSL_observation_period'])

# 9
fred_payems_data_df = fred_data_dict[2]
fred_payems_data_df['PAYEMS_observation_period'] = pd.to_datetime(fred_payems_data_df['PAYEMS_observation_period'])

# 10
fred_unrate_data_df = fred_data_dict[3]
fred_unrate_data_df['UNRATE_observation_period'] = pd.to_datetime(fred_unrate_data_df['UNRATE_observation_period'])

# <span style="color:brown"> Get all data sets combined

In [29]:
############# Merge data sets ##################################
def combine_all_data(stock_reset, data_reset):
    combined_data = stock_reset.copy()
    for df in data_reset:
        combined_data = pd.merge_asof(
        combined_data.sort_values('Date'),
        df.sort_values('Date'),
        left_on='Date',
        right_on='Date',
        direction='backward'
        )
    return combined_data.set_index('Date')
##################################################################

In [30]:
############# Merge each INDICATOR to the Stock Data using most recent data available on each date #############
def merge_asof_df(market_data, all_data_dict):
    data_combined = market_data.copy().reset_index()

    for indicator, data in all_data_dict.items():
        # Prepare right DataFrame
        data = data.copy().reset_index()
        if 'as_of_date_release' not in data.columns:
            raise KeyError(f"{indicator} is missing 'as_of_date_release' column")

        # Rename column to unique name to avoid collisions
        data = data.rename(columns={'as_of_date_release': f'{indicator}_as_of_date_release'})

        # --- Merge using most recent available data before each Date
        merged = pd.merge_asof(
            data_combined.sort_values('Date'),
            data.sort_values(f'{indicator}_as_of_date_release'),
            left_on='Date',
            right_on=f'{indicator}_as_of_date_release',
            direction='backward',
        )

        # --- Update combined frame for next iteration
        data_combined = merged.copy()

    return data_combined

################################################################################################

In [31]:
# FRED data dictionaries
dict_fred_names_data = {
    'fred_gdp_data_reset':    fred_gdp_data_df, 
    'fred_cpi_data_reset':    fred_cpi_data_df, 
    'fred_payems_data_reset': fred_payems_data_df,
    'fred_unrate_data_reset': fred_unrate_data_df, 
    'fred_fin_data_reset':    fred_fin_data_df
}

In [32]:
############# Ensure all DF have datetime index #############
def data_indx(df):
    data = df.copy()
    data.index = pd.to_datetime(data.index)
    data_reset = data.reset_index()
    return data_reset
#############################################################

In [33]:
# Load the news sentiment dta set
news_sentiment = pd.read_csv("project_DATA/news_sentiment_data.csv", index_col="date", parse_dates=True)

In [34]:
# Load the data sets prepared for merge
yf_sp500_data_index_reset = data_indx(yf_sp500_data_index_df)
yf_dxy_data_index_reset = data_indx(yf_dxy_data_index_df)
yf_vix_data_index_reset = data_indx(yf_vix_data_index_df)
fred_fin_data_reset = data_indx(fred_fin_data_df)    
news_sentiment_reset = data_indx(news_sentiment)

In [35]:
news_sentiment_reset.rename(columns={'date': 'Date', 'News Sentiment': 'news_sentiment'}, inplace=True)

In [36]:
# Create a list of features data (for data_combined_yf_fred_01)
data_indx_reset = [yf_dxy_data_index_reset, yf_vix_data_index_reset, fred_fin_data_reset, news_sentiment_reset]


### <span style="color:brown"> First data combined set (YF + FRED)

In [37]:
# Get a first data combined set: Target + Set of Features_01
data_combined_yf_fred_01 = combine_all_data(yf_sp500_data_index_reset, data_indx_reset)

### <span style="color:brown"> Second data combined set (YF + FRED)

In [38]:
# Create a dict of FRED-MACRO data sets (for data_combined_yf_fred_02)

fred_data_df_dict = {
    'GDP':      fred_gdp_data_df,
    'CPIAUCSL': fred_cpi_data_df,
    'PAYEMS':   fred_payems_data_df,
    'UNRATE':   fred_unrate_data_df
    }

In [39]:
# List of KEYs
fred_id_list = [*fred_data_df_dict]

In [40]:
# Use merge_asof for time-series alignment 
data_combined_yf_fred_02 = merge_asof_df(data_combined_yf_fred_01, fred_data_df_dict)

In [41]:
# Make a deep copy
data_combined_yf_fred = data_combined_yf_fred_02.copy(deep=True)
# Create a first list of features
base_features_list = data_combined_yf_fred.columns.tolist()

### <span style="color:green">  NEW FEATURES Release days for Economics data

In [42]:
################### Create a new feature: a binary indicator for release days ###################
def release_day(all_data, fred_id_list):
    data = all_data.copy()
    for fred_id in fred_id_list:
        data[f'is_{fred_id}_release_day'] = (
        data['Date'] == data[f'{fred_id}_as_of_date_release']
        ).astype(int)
    return data
###########################################################################################

In [43]:
KIN_features_df_01 = release_day(data_combined_yf_fred, fred_id_list)

In [44]:
# Make a deep copy and set inx as 'Date'
KIN_features_df_01.set_index('Date', inplace=True)

In [45]:
KIN_features_df_01.shape

(6539, 33)

In [46]:
########################### KIN ############################################

In [47]:
# Make a deep copy
KIN_features_df = KIN_features_df_01.copy(deep=True)

## <span style="color:green">   Create new features

In [48]:
############## Calculate On-Balance Volume (OBV) indicator ###################
def calculate_obv(data_df):
    """
    Calculate On-Balance Volume (OBV) indicator for base
    
    Parameters:
    df (pd.DataFrame): DataFrame containing:
        - 'Close', 'Volume' (for stock)  
    Returns:
    pd.DataFrame: DataFrame with 'OBV_base' and 'OBV_futures' columns added
    """
    df = data_df.copy(deep=True)
    
    # Base OBV calculation
    base_price_change = df['Close'].diff()
    base_obv_change = df['Volume'] / 10000
    base_obv_change[base_price_change < 0] = -base_obv_change[base_price_change < 0]
    base_obv_change[base_price_change == 0] = 0
    df['OBV_base'] = base_obv_change.cumsum()

    # Transform OBV (cumulative → changes)
    df['OBV_base_dif'] = df['OBV_base'].diff()
    
    return df
###############################################################################

In [49]:
KIN_features_df = calculate_obv(KIN_features_df)

## <span style="color:green">   Create new features

In [50]:
# New features
def new_features_fred_macro(data_df, fred_id_list):
    df = data_df.copy(deep=True)
    '''
    make sure '_observation_period' columns have a datetime format
    '''
    for indicator in fred_id_list:
        df[f'{indicator}_observation_period'] = pd.to_datetime(df[f'{indicator}_observation_period'])
        df[f'{indicator}_as_of_date_release'] = pd.to_datetime(df[f'{indicator}_as_of_date_release'])
    
    '''
    for CPI, PAYMES, UNRATE -> month
    for GDP -> quarter
    '''
    df['period_CPI'] = df['CPIAUCSL_observation_period'].dt.month
    df['period_PAYMES_UNRATE'] = df['PAYEMS_observation_period'].dt.month # The same day release
    df['period_GDP'] = df['GDP_observation_period'].dt.quarter

    df['CPI_pct()'] = df['CPIAUCSL_value'].pct_change()
    df['GDP_pct()'] = df['GDP_value'].pct_change()
    df['PAYEMS_pct()'] = df['PAYEMS_value'].pct_change()
    df['UNRATE_pct()'] = df['UNRATE_value'].pct_change()

    #=======================================#
    df['kin_f1_gdp_paym'] = df['PAYEMS_value'] /  df['GDP_value']
    df['kin_f2_unr_f1'] = df['UNRATE_value']/100 - df['kin_f1_gdp_paym']
    df['kin_f3_cpi_t10y3m'] = df['CPI_pct()']*100 - df['T10Y3M']
    #=======================================#

    return df

In [51]:
KIN_features_df_new = new_features_fred_macro(KIN_features_df, fred_id_list)

In [52]:
# Identify columns starting with 'Unnamed:'
unnamed_cols = [col for col in KIN_features_df_new.columns if 'Unnamed:' in col]
# Drop columns
KIN_features_df_new.drop(columns=unnamed_cols, inplace=True)
columns_to_drop = KIN_features_df_new.columns[KIN_features_df_new.columns.str.contains("as_of_date_release|observation")]
KIN_features_df_new.drop(columns=columns_to_drop, inplace=True)

In [53]:
KIN_features_df_new.shape

(6539, 37)

In [54]:
# Copy the data
KIN_features_df_new_copy = KIN_features_df_new.copy(deep=True)

## <span style="color:green">   Create new features

In [55]:
################################# Add OHLC featres #########################################
def add_ohlc_features(data_df, names):
    """
    Adds 13 candlestick/OHLC-based features to capture intraday dynamics
    """
    df = data_df.copy(deep=True)
    
    # ==========================================
    # 1. Basic OHLC returns
    # ==========================================
    for name in names:
        df[f'{name}_return'] = df[f'{name}'].pct_change()
    
    # ==========================================
    # 2. Gap Analysis (overnight moves)
    # ==========================================
    df['gap'] = (df['Open'] - df['Close'].shift(1)) / df['Close'].shift(1)
    # Gap down filled: gap > 0 and price recovered
    gap_down_filled = (df['gap'] > 0) & (df['Low'] < df['Open'])
    # Gap up filled: gap < 0 and price declined  
    gap_up_filled = (df['gap'] < 0) & (df['High'] > df['Open'])
    # Either gap filled
    df['gap_filled'] = (gap_down_filled | gap_up_filled).astype(int)
    
    # ==========================================
    # 3. Intraday Range (volatility proxy)
    # ==========================================
    df['daily_range'] = (df['High'] - df['Low']) / df['Close']
    df['range_pct_change'] = df['daily_range'].pct_change()
    
    # ==========================================
    # 4. Close Position in Range (strength)
    # ==========================================
    # 0 = closed at low, 1 = closed at high
    df['close_position'] = (df['Close'] - df['Low']) / (df['High'] - df['Low'] + 1e-10)  # Avoid div by 0
    
    # ==========================================
    # 5. Body and Wicks (candlestick analysis)
    # ==========================================
    df['body'] = (df['Close'] - df['Open']) / df['Close']  # Positive = bullish candle
    df['upper_wick'] = (df['High'] - df[['Open', 'Close']].max(axis=1)) / df['Close']
    df['lower_wick'] = (df[['Open', 'Close']].min(axis=1) - df['Low']) / df['Close']
    
    # ==========================================
    # 6. OHLC Relative Relationships
    # ==========================================
    df['high_close_ratio'] = df['High'] / df['Close']  # How much higher was intraday peak?
    df['low_close_ratio'] = df['Low'] / df['Close']    # How much lower was intraday trough?

    # ==========================================
    # Final cleanup
    # ==========================================    
    df = df.dropna()
    
    return df
###########################################################################################

In [56]:
names = {'Open', 'Close', 'High', 'Low', 'Volume',  'DXY_indx_close', 'VIX_indx_close',}
KIN_features_df_new_copy = add_ohlc_features(KIN_features_df_new_copy, names)

## <span style="color:green">   Create new features

In [57]:
####################################### Add TA featres ####################################
def add_technical_momentum_features(data_df):
    """
    Add technical indicators for short-term (5-15 day) predictions
    adds 19 TA features, loses ~50 rows
    """
    df = data_df.copy(deep=True)
    
    # ==========================================
    # 1. RSI (momentum)
    # ==========================================
    from ta.momentum import RSIIndicator
    df['RSI_14'] = RSIIndicator(df['Close'], window=14).rsi()
    df['RSI_7'] = RSIIndicator(df['Close'], window=7).rsi()
    
    # ADDED: Normalize RSI to [-1, 1] range for better scaling
    df['RSI_14_norm'] = (df['RSI_14'] - 50) / 50  # -1 (oversold) to +1 (overbought)
    df['RSI_7_norm'] = (df['RSI_7'] - 50) / 50
    
    # Drop raw RSI (keep normalized versions)
    df = df.drop(columns=['RSI_14', 'RSI_7'])
    
    # ==========================================
    # 2. MACD (trend)
    # ==========================================
    from ta.trend import MACD
    macd = MACD(df['Close'])
    df['MACD'] = macd.macd()
    df['MACD_signal'] = macd.macd_signal()
    df['MACD_diff'] = macd.macd_diff()
    
    # ADDED: Normalize MACD by price (make it scale-invariant)
    df['MACD_norm'] = df['MACD'] / df['Close']
    df['MACD_signal_norm'] = df['MACD_signal'] / df['Close']
    df['MACD_diff_norm'] = df['MACD_diff'] / df['Close']
    
    # Drop raw MACD (keep normalized versions)
    df = df.drop(columns=['MACD', 'MACD_signal', 'MACD_diff'])
    
    # ==========================================
    # 3. Bollinger Bands (volatility + mean reversion)
    # ==========================================
    from ta.volatility import BollingerBands
    bb = BollingerBands(df['Close'])
    df['BB_width'] = (bb.bollinger_hband() - bb.bollinger_lband()) / df['Close']
    df['BB_position'] = (df['Close'] - bb.bollinger_lband()) / (bb.bollinger_hband() - bb.bollinger_lband())
    
    # BB squeeze indicator (low volatility before breakout)
    df['BB_squeeze'] = (df['BB_width'] < df['BB_width'].rolling(20).mean()).astype(int)
    
    # ==========================================
    # 4. EMA - SMA (Moving Averages)
    # ==========================================
    # Short-term: EMA (minimal data loss)
    df['EMA_10'] = df['Close'].ewm(span=10, adjust=False).mean()
    df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()
    
    # Long-term: SMA (trend following)
    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['SMA_50'] = df['Close'].rolling(50).mean()
    
    # Derived features (relative to price)
    df['price_vs_EMA10'] = (df['Close'] - df['EMA_10']) / df['Close']
    df['price_vs_EMA20'] = (df['Close'] - df['EMA_20']) / df['Close']
    df['price_vs_SMA20'] = (df['Close'] - df['SMA_20']) / df['Close'] 
    df['price_vs_SMA50'] = (df['Close'] - df['SMA_50']) / df['Close']
    
    # Crossover signals (binary)
    df['EMA_cross'] = (df['EMA_10'] > df['EMA_20']).astype(int)
    df['MA_cross'] = (df['SMA_20'] > df['SMA_50']).astype(int)
    
    # Normalize trend_strength by price
    df['trend_strength_ema'] = (df['EMA_20'] - df['SMA_50']) / df['Close']
    
    # Drop raw MA columns (save memory)
    df = df.drop(columns=['EMA_10', 'EMA_20', 'SMA_20', 'SMA_50'])
    
    # ==========================================
    # 5. Volume momentum
    # ==========================================
    df['volume_SMA_20'] = df['Volume'].rolling(20).mean()
    df['volume_ratio'] = df['Volume'] / (df['volume_SMA_20'] + 1e-10)  # ← ADDED: Avoid div by 0
    
    # ADDED: Volume trend (is volume increasing or decreasing?)
    df['volume_trend'] = df['volume_SMA_20'].pct_change(5)
    
    # Drop raw volume SMA
    df = df.drop(columns=['volume_SMA_20'])
    
    # ==========================================
    # 6. Additional Momentum Indicators
    # ==========================================
    
    # ADDED: Rate of Change (momentum)
    df['ROC_10'] = ((df['Close'] - df['Close'].shift(10)) / df['Close'].shift(10))
    df['ROC_20'] = ((df['Close'] - df['Close'].shift(20)) / df['Close'].shift(20))
    
    # ADDED: ADX (trend strength)
    from ta.trend import ADXIndicator
    adx = ADXIndicator(df['High'], df['Low'], df['Close'], window=14)
    df['ADX'] = adx.adx()
    df['ADX_norm'] = df['ADX'] / 100  # Normalize to [0, 1]
    
    # Drop raw ADX
    df = df.drop(columns=['ADX'])
    # ==========================================
    # Final cleanup
    # ==========================================
    df = df.dropna()
    
    print(f"   Added {df.shape[1] - data_df.shape[1]} technical features")
    print(f"   Remaining data: {len(df)} rows (lost {len(data_df) - len(df)} rows)")
    
    return df
###########################################################################################

In [58]:
KIN_features_df_new_copy = add_technical_momentum_features(KIN_features_df_new_copy)

   Added 20 technical features
   Remaining data: 6489 rows (lost 49 rows)


In [59]:
################ Check if any NaN exists in the entire DataFrame ##########################
def check_nan(df):
    nan_mask = df.isna()
    has_nan = nan_mask.any().any()
    print(f"Does the DataFrame contain any NaN values? {has_nan}")
    if has_nan:
        nan_counts = nan_mask.sum()
        print(f"Count of NaN values per column:\n{nan_counts[nan_counts > 0]}")
###########################################################################################

In [60]:
check_nan(KIN_features_df_new_copy)

Does the DataFrame contain any NaN values? False


In [61]:
data_df = KIN_features_df_new_copy.copy(deep=True)

In [62]:
data_df.shape

(6489, 74)

In [63]:
################# 1. Data Preparation Function  #####################
def prepare_returns_features(data_df):
    """
    Prepare features for returns-based modeling
    Leverages existing _pct() columns and adds minimal transformations
    """
    df = data_df.copy(deep=True)
    
    # 1. Rename Close_pct() to target
    df = df.rename(columns={'Close_return': 'target'})
  
    # 2. Single list of columns to drop if they exist.
    #    Now we have more informative new features derivatives this raw data
    
    columns_to_drop = [
        'Open', 'High', 'Low', #'Volume', ---> keep it for the next Step to Create Market State Variables
        'DXY_indx_close', #'VIX_indx_close', ---> keep it for the next Step to Create Market State Variables
        'DGS10', 'T10Y2Y', 'DTB3',
        'GDP_value', 'CPIAUCSL_value', 'PAYEMS_value', 'UNRATE_value',
        'is_UNRATE_release_day', 'OBV_base'
        ]
  
    # Keep Close for price reconstruction
    close_prices = df['Close'].copy()
    
    # Drop raw levels
    features_df = df.drop(columns=['Close'] + columns_to_drop, errors='ignore')
     
    # 3. Move 'target' to the end
    cols = [col for col in features_df.columns if col != 'target'] + ['target']
    features_df = features_df[cols]

    # 4. Drop NaN from diff() operations
    features_df.dropna(inplace=True)
    close_prices = close_prices.loc[features_df.index]
    
    print(f"   Data prepared: {len(features_df.columns) - 1} features, {len(features_df)} samples")
    
    # Verify 'target' position 
    print(f"   Target at position: {features_df.columns.get_loc('target')} (last)")
    
    return features_df, close_prices
################################### 1.OPT ####################################################

In [64]:
features_df, cloce_prices = prepare_returns_features(data_df)

   Data prepared: 59 features, 6489 samples
   Target at position: 59 (last)


In [65]:
data_for_part_1_2_df = features_df.copy(deep=True)

# Save the data

In [66]:
data_for_part_1_2_df.to_csv("project_DATA/features_for_part_1_2_df.csv")

In [67]:
cloce_prices_df = pd.DataFrame(cloce_prices)

In [68]:
cloce_prices_df.to_csv("project_DATA/CLOSE_data_for_part_1_2.csv")

In [69]:
data_for_part_1_2_df.shape

(6489, 60)

In [70]:
data_for_part_1_2_df.columns.to_list()

['Volume',
 'VIX_indx_close',
 'term_spread',
 'T10Y3M',
 'T10Y2Y_yield_curve',
 'DGS10_dif',
 'T10Y2Y_dif',
 'DTB3_dif',
 'news_sentiment',
 'is_GDP_release_day',
 'is_CPIAUCSL_release_day',
 'is_PAYEMS_release_day',
 'OBV_base_dif',
 'period_CPI',
 'period_PAYMES_UNRATE',
 'period_GDP',
 'CPI_pct()',
 'GDP_pct()',
 'PAYEMS_pct()',
 'UNRATE_pct()',
 'kin_f1_gdp_paym',
 'kin_f2_unr_f1',
 'kin_f3_cpi_t10y3m',
 'VIX_indx_close_return',
 'Volume_return',
 'Low_return',
 'DXY_indx_close_return',
 'High_return',
 'Open_return',
 'gap',
 'gap_filled',
 'daily_range',
 'range_pct_change',
 'close_position',
 'body',
 'upper_wick',
 'lower_wick',
 'high_close_ratio',
 'low_close_ratio',
 'RSI_14_norm',
 'RSI_7_norm',
 'MACD_norm',
 'MACD_signal_norm',
 'MACD_diff_norm',
 'BB_width',
 'BB_position',
 'BB_squeeze',
 'price_vs_EMA10',
 'price_vs_EMA20',
 'price_vs_SMA20',
 'price_vs_SMA50',
 'EMA_cross',
 'MA_cross',
 'trend_strength_ema',
 'volume_ratio',
 'volume_trend',
 'ROC_10',
 'ROC_20',
 